# Colab notebook for fine-tuning

In [ ]:
%pip install transformers datasets pandas scikit-learn sentencepiece torch accelerate -q

In [ ]:
import pandas as pd
import logging
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
import numpy as np
import torch

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
LOGGER = logging.getLogger(__name__)

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
LOGGER.info(f"Using device: {device}")

MODEL_NAME = "google/flan-t5-small"
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128
TASK_PREFIX = "translate slang to standard: "

In [ ]:
from transformers import DataCollatorForSeq2Seq

In [ ]:
def load_data(train_path, test_path):
    """Load train/test CSVs into Hugging Face Datasets."""
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Drop rows with missing source or target text
    train_df = train_df.dropna(subset=["source_text", "target_text"])
    test_df = test_df.dropna(subset=["source_text", "target_text"])

    train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
    test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

    LOGGER.info(f"Loaded {len(train_dataset)} training examples")
    LOGGER.info(f"Loaded {len(test_dataset)} test examples")

    return train_dataset, test_dataset


def preprocess_function(examples, tokenizer):
    """Tokenize inputs with task prefix and targets as labels.

    Updated to use modern text_target argument to avoid AttributeError.
    """
    inputs = [TASK_PREFIX + str(ex) for ex in examples["source_text"]]
    targets = [str(t) for t in examples["target_text"]]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )

    # Tokenize targets using the modern text_target argument
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
train_dataset, test_dataset = load_data(
    "/content/data/processed/train.csv",
    "/content/data/processed/test.csv"
)

In [ ]:
LOGGER.info(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
LOGGER.info(f"Model loaded. Parameters: {model.num_parameters():,.0f}")

In [ ]:
LOGGER.info("Tokenizing training dataset...")
train_dataset = train_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names,
)

LOGGER.info("Tokenizing test dataset...")
test_dataset = test_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=test_dataset.column_names,
)

LOGGER.info(f"Train dataset columns: {train_dataset.column_names}")

# Sanity check: verify labels are NOT all -100
sample_labels = train_dataset[0]["labels"]
valid_tokens = [t for t in sample_labels if t != -100]
LOGGER.info(f"Sample label has {len(valid_tokens)} valid tokens (should be > 0): {valid_tokens[:10]}")
LOGGER.info("Tokenization complete!")

Let's inspect a few samples from the tokenized `train_dataset` to understand the `input_ids` and `labels` content. This will help determine if the data is being correctly prepared for training.

In [ ]:
LOGGER.info("Inspecting tokenized training data samples...")

# Display the first 3 samples from the tokenized train_dataset
for i in range(3):
    sample = train_dataset[i]
    print(f"--- Sample {i+1} ---")
    print(f"Input IDs: {sample['input_ids']}")
    print(f"Attention Mask: {sample['attention_mask']}")
    print(f"Labels (with -100 for padding): {sample['labels']}")

    # Decode input_ids and labels for better understanding
    decoded_input = tokenizer.decode([id for id in sample['input_ids'] if id != tokenizer.pad_token_id], skip_special_tokens=True)
    # Filter out -100 before decoding labels
    decoded_label = tokenizer.decode([id for id in sample['labels'] if id != -100 and id != tokenizer.pad_token_id], skip_special_tokens=True)
    print(f"Decoded Input: {decoded_input}")
    print(f"Decoded Label: {decoded_label}")
    print("\n")

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="backend/fine_tuned_model",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    save_total_limit=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,
    learning_rate=5e-4,   # Bumped slightly — 1e-4 is on the low end for flan-t5-small fine-tuning
    seed=42,
    fp16=device == "cuda",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

LOGGER.info("Training configuration ready")

In [ ]:
# DataCollatorForSeq2Seq pads inputs and labels to the longest sequence in each
# batch, and replaces label padding tokens with -100 so they're ignored in loss.
# This is the ONLY place padding + masking should happen.
data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if device == "cuda" else None,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
)

LOGGER.info("Starting training...")
trainer.train()

In [ ]:
output_dir = "backend/fine_tuned_model"
import shutil

# Clear existing output directory to avoid file locking issues
if Path(output_dir).exists():
    shutil.rmtree(output_dir)

LOGGER.info(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir, safe_serialization=False)
LOGGER.info("Model and tokenizer saved!")
LOGGER.info("Training complete!")

In [ ]:
# Zip and download the fine-tuned model checkpoint
from google.colab import files
import shutil

model_checkpoint = "backend/fine_tuned_model"
shutil.make_archive("checkpoint", "zip", ".", model_checkpoint)
files.download("checkpoint.zip")
LOGGER.info("Checkpoint zipped and ready for download to Google Drive")

In [ ]:
# Load the trained model for inference
trained_model = AutoModelForSeq2SeqLM.from_pretrained(output_dir)
trained_tokenizer = AutoTokenizer.from_pretrained(output_dir)
trained_model.eval()

# Test example
test_input = "wanna chill, im done grinding."
inputs = trained_tokenizer(test_input, return_tensors="pt")
with torch.no_grad():
    outputs = trained_model.generate(**inputs, max_length=128)
result = trained_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Input:  {test_input}")
print(f"Output: {result}")